In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../data/experiment_data.csv")

required = {
    "user_id", "variant", "converted",
    "revenue", "contribution_profit"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if df["user_id"].duplicated().any():
    raise ValueError("Duplicate user IDs found. Investigate before testing.")

if set(df["variant"].dropna().unique()) != {"Control", "Treatment"}:
    raise ValueError("Expected exactly Control and Treatment groups.")

if df[list(required)].isna().any().any():
    raise ValueError("Missing values in required columns. Investigate before testing.")

if not df["converted"].isin([0, 1]).all():
    raise ValueError("'converted' must contain only 0 and 1.")

control = df[df["variant"] == "Control"].copy()
treatment = df[df["variant"] == "Treatment"].copy()

print("Control users:", len(control))
print("Treatment users:", len(treatment))
print("Data validation passed.")

Control users: 25000
Treatment users: 25000
Data validation passed.


In [3]:
# Test A conversion test

n_c = len(control)
n_t = len(treatment)

x_c = int(control["converted"].sum())
x_t = int(treatment["converted"].sum())

p_c = x_c / n_c
p_t = x_t / n_t

# Difference in conversion rates: Treatment - Control
lift_abs = p_t - p_c
lift_relative = lift_abs / p_c if p_c != 0 else np.nan

# Two-proportion z-test
z_stat, p_value_conversion = proportions_ztest(
    count=np.array([x_t, x_c]),
    nobs=np.array([n_t, n_c]),
    alternative="two-sided"
)

# Approximate 95% CI for the difference in proportions.
# With these large sample sizes, this is a useful first approximation.
se_diff = np.sqrt(
    p_t * (1 - p_t) / n_t +
    p_c * (1 - p_c) / n_c
)
z_critical = stats.norm.ppf(0.975)

ci_low = lift_abs - z_critical * se_diff
ci_high = lift_abs + z_critical * se_diff

# Cohen's h: effect size for two proportions
cohens_h = proportion_effectsize(p_t, p_c)

print("CONVERSION RATE TEST")
print(f"Control:       {p_c:.4%} ({x_c:,}/{n_c:,})")
print(f"Treatment:     {p_t:.4%} ({x_t:,}/{n_t:,})")
print(f"Absolute lift: {lift_abs * 100:.3f} percentage points")
print(f"Relative lift: {lift_relative:.2%}")
print(f"Z-statistic:   {z_stat:.4f}")
print(f"P-value:       {p_value_conversion:.6g}")
print(f"95% CI:        [{ci_low * 100:.3f}, {ci_high * 100:.3f}] percentage points")
print(f"Cohen's h:     {cohens_h:.4f}")

CONVERSION RATE TEST
Control:       8.7960% (2,199/25,000)
Treatment:     9.6680% (2,417/25,000)
Absolute lift: 0.872 percentage points
Relative lift: 9.91%
Z-statistic:   3.3679
P-value:       0.000757473
95% CI:        [0.365, 1.379] percentage points
Cohen's h:     0.0301


In [4]:
# Test B Revenue per assigned user

def welch_analysis(control_values, treatment_values):
    c = np.asarray(control_values, dtype=float)
    t = np.asarray(treatment_values, dtype=float)

    n_c, n_t = len(c), len(t)
    mean_c, mean_t = c.mean(), t.mean()
    var_c, var_t = c.var(ddof=1), t.var(ddof=1)

    difference = mean_t - mean_c

    # Welch standard error
    se = np.sqrt(var_t / n_t + var_c / n_c)

    # Welch-Satterthwaite degrees of freedom
    numerator = (var_t / n_t + var_c / n_c) ** 2
    denominator = (
        (var_t / n_t) ** 2 / (n_t - 1) +
        (var_c / n_c) ** 2 / (n_c - 1)
    )
    df_welch = numerator / denominator

    t_stat = difference / se
    p_value = 2 * stats.t.sf(abs(t_stat), df=df_welch)

    critical = stats.t.ppf(0.975, df=df_welch)
    ci_low = difference - critical * se
    ci_high = difference + critical * se

    # Cohen's d using pooled SD as a standardized descriptive effect size
    pooled_sd = np.sqrt(
        ((n_c - 1) * var_c + (n_t - 1) * var_t) /
        (n_c + n_t - 2)
    )
    d = difference / pooled_sd if pooled_sd > 0 else np.nan

    return {
        "control_mean": mean_c,
        "treatment_mean": mean_t,
        "difference": difference,
        "relative_lift": difference / mean_c if mean_c != 0 else np.nan,
        "t_statistic": t_stat,
        "degrees_of_freedom": df_welch,
        "p_value": p_value,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "cohens_d": d
    }

revenue_result = welch_analysis(
    control["revenue"],
    treatment["revenue"]
)

print("REVENUE PER USER — WELCH'S T-TEST")
for key, value in revenue_result.items():
    print(f"{key}: {value:.6g}")

REVENUE PER USER — WELCH'S T-TEST
control_mean: 81.5231
treatment_mean: 86.7226
difference: 5.1995
relative_lift: 0.0637794
t_statistic: 1.95284
degrees_of_freedom: 49988.6
p_value: 0.0508442
ci_low: -0.0190958
ci_high: 10.4181
cohens_d: 0.0174667


In [5]:
# Test C Contribution profit per assigned user

profit_result = welch_analysis(
    control["contribution_profit"],
    treatment["contribution_profit"]
)

print("CONTRIBUTION PROFIT PER USER — WELCH'S T-TEST")
for key, value in profit_result.items():
    print(f"{key}: {value:.6g}")

CONTRIBUTION PROFIT PER USER — WELCH'S T-TEST
control_mean: 23.2733
treatment_mean: 24.785
difference: 1.5117
relative_lift: 0.0649542
t_statistic: 1.94135
degrees_of_freedom: 49977.3
p_value: 0.0522213
ci_low: -0.0145303
ci_high: 3.03793
cohens_d: 0.017364


In [6]:
results = {
    "Conversion rate": p_value_conversion,
    "Revenue per user": revenue_result["p_value"],
    "Contribution profit per user": profit_result["p_value"]
}

metric_names = list(results.keys())
raw_p_values = list(results.values())

reject, adjusted_p_values, _, _ = multipletests(
    raw_p_values,
    alpha=0.05,
    method="holm"
)

decision_table = pd.DataFrame({
    "Metric": metric_names,
    "Raw_P_Value": raw_p_values,
    "Holm_Adjusted_P_Value": adjusted_p_values,
    "Significant_After_Correction": reject
})

print(decision_table.to_string(index=False))

                      Metric  Raw_P_Value  Holm_Adjusted_P_Value  Significant_After_Correction
             Conversion rate     0.000757               0.002272                          True
            Revenue per user     0.050844               0.101688                         False
Contribution profit per user     0.052221               0.101688                         False


In [7]:
n_users = len(treatment)

incremental_profit_per_user = profit_result["difference"]
estimated_incremental_profit = incremental_profit_per_user * n_users

profit_ci_low = profit_result["ci_low"] * n_users
profit_ci_high = profit_result["ci_high"] * n_users

print("FINANCIAL IMPACT — SAME NUMBER OF USERS")
print(f"Incremental profit per user: {incremental_profit_per_user:.2f}")
print(f"Estimated incremental profit: {estimated_incremental_profit:,.2f}")
print(
    f"Approx. 95% CI for incremental profit: "
    f"[{profit_ci_low:,.2f}, {profit_ci_high:,.2f}]"
)

FINANCIAL IMPACT — SAME NUMBER OF USERS
Incremental profit per user: 1.51
Estimated incremental profit: 37,792.51
Approx. 95% CI for incremental profit: [-363.26, 75,948.28]


In [8]:
final_summary = pd.DataFrame([
    {
        "Metric": "Conversion rate",
        "Control": p_c,
        "Treatment": p_t,
        "Treatment_minus_Control": lift_abs,
        "P_Value": p_value_conversion
    },
    {
        "Metric": "Revenue per user",
        "Control": revenue_result["control_mean"],
        "Treatment": revenue_result["treatment_mean"],
        "Treatment_minus_Control": revenue_result["difference"],
        "P_Value": revenue_result["p_value"]
    },
    {
        "Metric": "Contribution profit per user",
        "Control": profit_result["control_mean"],
        "Treatment": profit_result["treatment_mean"],
        "Treatment_minus_Control": profit_result["difference"],
        "P_Value": profit_result["p_value"]
    }
])

print(final_summary.round(4).to_string(index=False))

                      Metric  Control  Treatment  Treatment_minus_Control  P_Value
             Conversion rate   0.0880     0.0967                   0.0087   0.0008
            Revenue per user  81.5231    86.7226                   5.1995   0.0508
Contribution profit per user  23.2733    24.7850                   1.5117   0.0522


## Business Recommendation & Conclusion

### 1. Key Findings

| Metric | Control | Treatment | Result |
|---|---:|---:|---|
| Conversion Rate | 8.80% | 9.67% | Significant improvement |
| Revenue per User | 81.52 | 86.72 | Not statistically significant |
| Profit per User | 23.27 | 24.79 | Not statistically significant |

### 2. Business Insights

- Conversion rate increased by **9.91%**, and the result remained significant after multiple-testing correction.
- Revenue per user increased by **6.38%**, but the improvement is not statistically significant.
- Profit per user increased by **6.50%**, but the evidence is insufficient to confirm a real improvement.
- Estimated additional profit is approximately **37,793 units** for 25,000 users, but the confidence interval includes zero.

### 3. Business Recommendation

**Recommendation: Continue testing before a full rollout.**

The new Buy Now button improves conversion, but its impact on revenue and profit is still uncertain. I would run a follow-up experiment with profit per user as the primary metric before making a final decision.

### 4. Conclusion

This experiment shows that improving conversion does not necessarily guarantee higher profitability. The treatment increased conversion significantly, but the financial improvements were not statistically significant after correction. Therefore, more evidence is needed before confirming that the new design is better for the business.

*Note: This is a simulated dataset created for analysis and learning purposes.*